# SmolVLM-256M: EVLM Benchmark

This notebook runs the benchmark of the EVLM paper (Farahani et al., *EVLM: Intent-Driven Edge Vision Language Model for UAV-Based Power Line Inspection*, EDGE 2026) with the fastest inference setup of this repository.

The paper uses two public datasets (Section V-B):

- **Power line videos** (Xing et al., IROS 2023, ref. [35]): 720x480 frames at 30 fps from eight environments. The paper cuts them into 4 s clips of 120 frames. This dataset is used for the speed tests.
- **InsPLAD** (Vieira e Silva et al., 2023, ref. [43]): UAV images of power line parts with labels. We use the fault classification crops (good, rust and other defects). This dataset is used for the accuracy test.

`get_data.py` downloads only the parts that the test needs: one 4 s clip for each environment (about 0.5 GB) and the InsPLAD fault crops (about 0.9 GB). The full archives are 25 GB and 6.4 GB.

`evlm_eval.py` has two tasks:

1. `clips`: for each clip, a histogram filter keeps the key frames (512 bins, threshold 0.95, as in the paper). The VLM answers the operator intent for each key frame. The script prints the key frames, the filter time, the VLM time and the segment latency.
2. `insplad`: the VLM answers the two prompts of the paper's Table III for each image. The script scores the component name and the rust answer against the labels.

The defaults are the best setup from the earlier tests: fast GPU preprocessing, `--size 1024`, CUDA-graph decoding, stop at EOS, and a TensorRT engine for SigLIP. Each part has a flag, so the last cell can turn the parts off one at a time (ablation).

We use the base model. The paper fine-tunes it with LoRA, and its weights are not public, so the accuracy here is lower than in the paper.

In [3]:
!git -C /content/edge-ai-smol-lvm-250-optimization pull
!pwd

fatal: cannot change to '/content/edge-ai-smol-lvm-250-optimization': No such file or directory
/content


In [4]:
!git clone https://github.com/mahdikhashan/edge-ai-smol-lvm-250-optimization /content/edge-ai-smol-lvm-250-optimization

Cloning into '/content/edge-ai-smol-lvm-250-optimization'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 257 (delta 167), reused 156 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 155.13 KiB | 7.76 MiB/s, done.
Resolving deltas: 100% (167/167), done.


In [5]:
REPO = "./edge-ai-smol-lvm-250-optimization"
DATA = f"{REPO}/data"
OUT = f"{REPO}/results"

# Colab: packages for the TensorRT engine. The Jetson has TensorRT in JetPack.
!pip install -q tensorrt onnx

# dataset setup: about 1.4 GB, only the files the test needs; files that exist already are skipped
!python {REPO}/get_data.py --out {DATA}

clips: forest (powerline_1_forest) clip_0: 120 frames from frame 11295
clips: city1 (powerlines_16_city) clip_0: 120 frames from frame 21107
clips: mountain (powerline_7_mountain) clip_0: 120 frames from frame 19277
clips: city2 (powerline_5_japan) clip_0: 120 frames from frame 16784
clips: sand (powerline_3_sand) clip_0: 120 frames from frame 14129
clips: warehouse (powerline_11_warehouse) clip_0: 120 frames from frame 27219
clips: wasteland (powerline_4_wasteland) clip_0: 120 frames from frame 15424
clips: industrial (powerline_2_industrial) clip_0: 120 frames from frame 12632
insplad: extracted to edge-ai-smol-lvm-250-optimization/data/insplad


In [6]:
# speed test on the power line clips, best setup
!python {REPO}/evlm_eval.py --task clips --data {DATA} --out {OUT}

processor_config.json: 100% 68.0/68.0 [00:00<00:00, 391kB/s]
chat_template.json: 100% 429/429 [00:00<00:00, 1.92MB/s]
preprocessor_config.json: 100% 486/486 [00:00<00:00, 2.99MB/s]
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
config.json: 100% 7.35k/7.35k [00:00<00:00, 23.0MB/s]
tokenizer_config.json: 100% 28.2k/28.2k [00:00<00:00, 83.6MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 34.2MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 109MB/s]
added_tokens.json: 100% 4.74k/4.74k [00:00<00:00, 18.7MB/s]
special_tokens_map.json: 100% 1.07k/1.07k [00:00<00:00, 3.83MB/s]
tokenizer.json: 100% 3.55M/3.55M [00:00<00:00, 186MB/s]

model.safetensors: downloading bytes:  33% 169M/513M [00:01<00:01, 231MB/s, 13.4MB/s  ]  
model.safetensors: downloading bytes:  45% 229M/513M [00:01<00:01, 258MB/s, 18.1MB/s  ]
model.safetensors: downloading bytes:  5

In [7]:
# accuracy test on InsPLAD (10 images for each asset and label), best setup
!python {REPO}/evlm_eval.py --task insplad --data {DATA} --out {OUT} --limit 10

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100% 471/471 [00:01<00:00, 269.33it/s]
[transformers] LANCZOS resample requires torchvision >= 0.27 and processing on CPU; it is not supported on CUDA or for torchvision < 0.27. Falling back to BICUBIC which approximates LANCZOS. To match the original model exactly, upgrade torchvision and move the tensors to CPU before resizing, or use a PIL backend image processor instead.
[10/23/2026-20:43:36] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[10/23/2026-20:43:44] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So th

In [8]:
# ablation: turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/key':>10} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_key']:10.1f} {s['segment_s']:9.2f}")

config               key frames VLM ms/key segment s
best                       1.00      240.1      1.97
no TensorRT                1.00      251.7      1.85
default tiles              1.00      408.0      2.05
no stop at EOS             1.00      510.1      2.12
eager decode               1.00      805.5      2.35
bare generate              1.00      847.4      2.44
slow preprocess            1.00      352.5      1.93
no frame filter          120.00      191.9     24.24
nothing (baseline)         1.00     1367.2      2.92


In [9]:
# ablation (new version: tokens and ms per token): turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/call':>11} {'tokens':>6} {'ms/token':>8} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call']:11.1f} {s['tokens_per_answer']:6.1f} "
          f"{s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f}")

config               key frames VLM ms/call tokens ms/token segment s
best                       1.00       221.4   19.5     6.90      0.54
no TensorRT                1.00       239.8   19.5     6.86      0.57
default tiles              1.00       389.5   19.5     8.26      0.71
no stop at EOS             1.00       506.6   19.5    21.26      0.84
eager decode               1.00       794.6   19.5    34.21      1.11
bare generate              1.00       829.5   19.5    36.04      1.16
slow preprocess            1.00       351.7   19.5     8.02      0.68
no frame filter          120.00       187.8   14.1     6.98     22.54
nothing (baseline)         1.00      1354.7   19.5    33.87      1.69


## Comparison with the paper (run on the Jetson Orin)

The paper ran on an NVIDIA Jetson (8-core CPU, 16 GB, 1024 CUDA cores, 32 Tensor cores). Run the next cell on that device. Times from another GPU cannot be compared with the paper.

The cell runs two setups on all 8 environments:

- `paper-like baseline`: the plain model path (slow preprocessing, default tiles, Hugging Face `generate`, no TensorRT). This is our closest copy of the paper's pipeline. The paper does not give its inference code.
- `best`: all the speed-ups.

Compare the time for each VLM call, not the time for each segment. The paper's filter keeps about 3.45 frames per segment, and our clips can keep a different number. The last column scales our segment to 3.45 key frames.

The paper's time for each call is an estimate: 6.448 s per segment / 3.45 calls = about 1.87 s.

Also compare the tokens per answer, because a longer answer takes more time. The paper uses a fine-tuned model, so its answers can be longer or shorter.

The segment time is the filter time plus the VLM calls. The PNG decode is not included, because the paper decodes video and we decode PNG files.

In [ ]:
# Orin comparison: run on the Jetson Orin, all 8 environments
import json
import subprocess

PAPER_SEGMENT_S, PAPER_KEY_FRAMES = 6.448, 3.45  # Table V (EVLM) and the FOT frames per segment
SETUPS = {
    "paper-like baseline": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
    "best": [],
}

results = {}
for name, flags in SETUPS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name}: failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    results[name] = json.loads(line[len("SUMMARY "):])

print(f"{'setup':<20} {'key frames':>10} {'VLM s/call':>10} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8} "
      f"{'segment s':>9} {'segment s at 3.45 keys':>22}")
for name, s in results.items():
    at_paper = s["filter_s_per_segment"] + PAPER_KEY_FRAMES * s["vlm_ms_per_call"] / 1e3
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call'] / 1e3:10.2f} {s['prefill_ms']:10.1f} "
          f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f} {at_paper:22.2f}")
print(f"{'paper (EVLM)':<20} {PAPER_KEY_FRAMES:10.2f} {PAPER_SEGMENT_S / PAPER_KEY_FRAMES:10.2f} {'-':>10} {'-':>6} "
      f"{'-':>8} {PAPER_SEGMENT_S:9.2f} {PAPER_SEGMENT_S:22.2f}")

if len(results) == 2:
    base, best = results["paper-like baseline"], results["best"]
    print(f"speed-up for each VLM call, best against baseline: {base['vlm_ms_per_call'] / best['vlm_ms_per_call']:.2f}x")

In [3]:
# llama.cpp (GGUF) on both EVLM tasks, every quant. The PyTorch best setup runs too, as the reference.
# Colab: builds llama.cpp with CUDA if it is missing (about 20-30 minutes; -j 2 avoids the out-of-memory kill).
import json
import os
import subprocess

print("repo:", REPO, "| data:", DATA)
# Colab: the packages the PyTorch path needs, and the datasets when they are missing (a fresh clone has no data/).
!pip install -q tensorrt onnx
if not (os.path.isdir(f"{DATA}/powerline_clips") and os.path.isdir(f"{DATA}/insplad")):
    subprocess.run(["python", f"{REPO}/get_data.py", "--out", DATA], check=True)

BIN = "./llama.cpp/build/bin"
subprocess.run(
    "test -f ./llama.cpp/build/bin/llama-mtmd-cli || ("
    "git clone --depth 1 https://github.com/ggml-org/llama.cpp && "
    "cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && "
    "cmake --build llama.cpp/build --config Release -j 2 --target llama-mtmd-cli llama-quantize)",
    shell=True, check=True,
)

CONFIGS = [("PyTorch best", [])] + [
    (f"GGUF {q}", ["--backend", "gguf", "--quant", q, "--cli", f"{BIN}/llama-mtmd-cli",
                   "--quantize", f"{BIN}/llama-quantize", "--gguf-dir", f"{REPO}/gguf"])
    for q in ("f16", "q8_0", "q4_k_m")
]

summaries = {}
for task in ("clips", "insplad"):
    for name, flags in CONFIGS:
        r = subprocess.run(["python", f"{REPO}/evlm_eval.py", "--task", task, "--data", DATA, "--out", OUT, *flags],
                           capture_output=True, text=True)
        line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
        if line is None:
            print(f"{task} {name}: failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
            continue
        summaries[(task, name)] = json.loads(line[len("SUMMARY "):])

print("clips (8 environments, 4 s segments)")
print(f"{'setup':<14} {'key frames':>10} {'VLM ms/call':>11} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8} {'segment s':>9}")
for name, _ in CONFIGS:
    s = summaries.get(("clips", name))
    if s:
        print(f"{name:<14} {s['key_frames']:10.2f} {s['vlm_ms_per_call']:11.1f} {s['prefill_ms']:10.1f} "
              f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f}")

print("\ninsplad (110 images, component and rust prompts)")
print(f"{'setup':<14} {'component acc':>13} {'rust acc':>8} {'VLM ms/call':>11} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8}")
for name, _ in CONFIGS:
    s = summaries.get(("insplad", name))
    if s:
        rust = f"{s['rust_acc']:8.3f}" if s["rust_acc"] is not None else f"{'-':>8}"
        print(f"{name:<14} {s['component_acc']:13.3f} {rust} {s['vlm_ms_per_call']:11.1f} {s['prefill_ms']:10.1f} "
              f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f}")

NameError: name 'REPO' is not defined